# Stage 2: CyTOF preprocessing

Load the split CSVs, apply histone gates, transform marker values, correct batch effects, normalize, and z-score marker values.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))

# Add the repository's vendored correction and normalization libraries.
sys.path.insert(0, str(PROJECT_ROOT / "external" / "som_ot_warp_repo_anchor"))
sys.path.insert(0, str(PROJECT_ROOT / "external" / "cytof_helper"))
import app  # noqa: E402

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")
print(f"Kernel executable: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"Imported app: {Path(app.__file__).resolve()}")

In [ ]:
from app.config.config import EXPERIMENTS, METADATA_COLUMNS
from app.config import (
    OUTPUT_ROOT,
    SKIP_BATCH_CORRECTION,
    SUBSAMPLE,
    DEMO_MODE,
    DATA_DIR,
    ANNOTATION_FILE,
)
from app.config.config import CELL_IDENTITY_MARKERS, EPIGENETIC_MARKERS
from app.utils import attach_annotation_labels

from app.preprocessing import (
    arcsinh_transform,
    gate_histones,
    hierarchical_zscore,
    pool_csv_files,
    subsample_gfp_high,
    normalize_pooled_dataframe_new,
)

In [ ]:
INPUT_DIR = DATA_DIR / "3. GFP split after CD45 exclusion"

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {INPUT_DIR}")
print(f"Demo mode: {DEMO_MODE}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_ROOT}")
use_labels = True

## Pool and filter GFP-high cells

- Read and concatenate CSV files from the configured GFP folders, preserving each source path in `path`.
- Drop the unused input columns listed in the pooling cell.
- Identify non-metadata columns that contain only missing values in any experiment/sample, then drop those columns from the pooled table.
- Keep only GFP-high cells, using the source folder name to identify the GFP group.

In [ ]:
gfp_csv_files = sorted(
    path
    for path in INPUT_DIR.rglob("*.csv")
    if path.name not in ["GFP_split_summary.csv", "pooled_gfp_df.csv"]
)
pooled_gfp_df, missing_columns_by_sample = pool_csv_files(
    gfp_csv_files,
    source_column="path",
    drop_columns=[
        "Line",
        "cell_id",
        "replicate",
        "source_file",
        "source_path",
        "annotation_present",
        "batch",
        # 'index',
    ],
    metadata_columns=METADATA_COLUMNS,
)
if not pooled_gfp_df["cell_uid"].is_unique:
    raise ValueError("Pooled cell_uid values must be unique.")


missing_metadata_columns = sorted(set(METADATA_COLUMNS) - set(pooled_gfp_df.columns))
if missing_metadata_columns:
    raise ValueError(f"Pooled input is missing metadata columns: {missing_metadata_columns}")

print("Pooled GFP table shape:", pooled_gfp_df.shape)

pooled_gfp_df["gfp_group"] = pooled_gfp_df["path"].map(lambda path: Path(path).parent.name)
pooled_gfp_df = pooled_gfp_df.copy().loc[pooled_gfp_df["gfp_group"] == "GFP_high"].copy()
pooled_gfp_df.drop(["gfp_group"], inplace=True, axis=1)
if pooled_gfp_df.empty:
    raise ValueError("No GFP_high cells are available.")


missing_columns = missing_columns_by_sample.keys()
print("missing Non-metadata columns:", missing_columns)
pooled_gfp_df.drop(missing_columns, inplace=True, axis=1)

# load annotations
pooled_gfp_df = attach_annotation_labels(
    df=pooled_gfp_df.copy(), annotation_file=str(ANNOTATION_FILE)
)


pooled_gfp_df.reset_index(drop=True, inplace=True)
print("Pooled GFP High table shape:", pooled_gfp_df.shape)

# pooled_gfp_df.to_csv(INPUT_ROOT/"pooled_gfp_df.csv", index=False)
print("table columns:", pooled_gfp_df.columns)

In [ ]:
treated = np.mean(pooled_gfp_df[pooled_gfp_df["condition"] == "treated"]["GAL4"])
untreated = np.mean(pooled_gfp_df[pooled_gfp_df["condition"] == "untreated"]["GAL4"])

treated = np.mean(pooled_gfp_df[pooled_gfp_df["condition"] == "treated"]["TSPAN8"])
untreated = np.mean(pooled_gfp_df[pooled_gfp_df["condition"] == "untreated"]["TSPAN8"])
print(treated, untreated)
print(treated, untreated)

## 1. Apply raw histone gates

Keep rows where raw `H3.3` and `H4` both exceed 5. Within each mouse, remove rows with any numeric marker outside the 0.01% to 99.99% quantile bounds, calculated per `sample_name`, `experiment`, and `condition`. `df_gated` contains the retained rows and preserves the input columns.

In [ ]:
df_gated, gated_data_columns = gate_histones(pooled_gfp_df.copy(), skip_columns=METADATA_COLUMNS)
# if set(df_gated.columns) != set(gated_data_columns + METADATA_COLUMNS):
#     print("ERROR: histone gating changed the dataframe columns.")
print("Pooled gated table shape:", df_gated.shape)

## 2. Apply SOM-OT batch correction

When `SKIP_BATCH_CORRECTION` is true, pass the ArcSinh-transformed table through unchanged. 

Otherwise, fit the SOM-OT model using `df_arcsinh`, then transform its rows using their experiment labels. Shared markers are corrected in one transform call; nonshared or unsupported marker columns are omitted from `df_batch_corrected`.

In [ ]:
if SKIP_BATCH_CORRECTION:
    df_batch_corrected = df_gated
    print("skipped batch correction ")
else:
    df_to_correct = df_gated.copy()

    # # =========================
    # # CREATE MODEL

    from somot_warp.multi_batch import MultiBatchSOMOTWarp

    # =========================
    # BATCH CORRECTION FUNCTION
    # =========================
    def create_xy(df, features, y_col="experiment"):
        X = df[features].to_numpy(dtype=float)
        y = df[y_col].replace({"batch2": 2, "batch3": 3, "batch4_extra_treatment": 4})
        y = y.to_numpy()

        # Validate model input
        if not np.isfinite(X).all():
            raise ValueError("X contains NaN or Inf values")

        print("\nClean X shape:", X.shape)

        print("NaN values:", np.isnan(X).sum())
        print("Inf values:", np.isinf(X).sum())
        print("Minimum:", X.min())
        print("Maximum:", X.max())
        return X, y

    # =========================
    # MARKERS
    # =========================

    batch_coorection_cols = [
        "CD109",
        "CD24",
        "CD44",
        "CD45",
        "CXCR4",
        "E-cadherin",
        "EZH2",
        "EpCAM",
        "GAL4",
        "GATA6",
        "GFP",
        "H2AK119ub",
        "H3",
        "H3.3",
        "H3K27ac",
        "H3K27me3",
        "H3K36me2",
        "H3K36me3",
        "H3K4me1",
        "H3K4me2",
        "H3K4me3",
        "H3K9ac",
        "H3K9me2",
        "H3K9me3",
        "H4",
        "H4K16ac",
        "H4K20me3",
        "KI67",
        "KRT17",
        "KRT5",
        "MBD",
        "MHC-II",
        "Pan-KRT",
        "S100A4",
        "TSPAN8",
        "Vimentin",
        "pH2A.X",
        "pRb",
    ]

    # All biological markers to standardize across experiments
    # Includes markers used downstream for UMAP
    all_marker_cols = list(
        dict.fromkeys(batch_coorection_cols + CELL_IDENTITY_MARKERS + EPIGENETIC_MARKERS)
    )
    import numpy as np
    import pandas as pd

    # 1. Intersect master list with columns present in the DataFrame
    shared_markers = set(all_marker_cols).intersection(df_to_correct.columns)

    # 2. Filter out markers containing NaN or Inf values
    valid_markers = set(shared_markers)

    for marker in list(valid_markers):
        values = pd.to_numeric(df_to_correct[marker], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(values).all():
            valid_markers.remove(marker)

    # 3. Produce the final cleaned list
    marker_cols_ = [m for m in batch_coorection_cols if m in valid_markers]

    print("Final marker_cols_:", marker_cols_)
    print("Count:", len(marker_cols_))

    # --------------------

    def create_subset(fit_df):
        fit_subset = (
            fit_df.groupby(["experiment", "condition"], group_keys=False)
            .sample(frac=1, random_state=42)
            .groupby(["experiment", "condition"], group_keys=False)
            .head(10000)
        )
        return fit_subset

    def batch_correct(
        df_corrected,
        marker_cols_,
        #   batch_label, invalid_markers=None
    ):

        X, y = create_xy(df=df_corrected.copy(), features=marker_cols_)

        # Apply fitted batch-correction model
        X_corrected = model.transform(X, y)

        if not np.isfinite(X_corrected).all():
            raise ValueError(f"Batch correction produced NaN or Inf f")

        # Convert corrected values back to arcsinh scale
        df_corrected[marker_cols_] = X_corrected

        return df_corrected

    # # =========================
    # # FIT MODEL
    # # =========================
    # df = df_gated.copy()
    model = MultiBatchSOMOTWarp(
        anchor_label=2,  # batch2
        n_jobs=-1,
        som_verbose=True,
        pca_components=None,
        transform_kind="arcsinh",
        n_landmarks_total=1200,
        som_grid=(8, 8),
        q_grid=np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]),
        alpha_ot=0.7,
        alpha_warp=None,
    )

    X_fit, y_fit = create_xy(
        df=create_subset(df_to_correct.copy()),
        features=marker_cols_,
    )

    model.fit(X_fit, y_fit)

    print("\nBatch-correction model fitted")

    df_batch_corrected = batch_correct(
        df_to_correct.copy(),
        marker_cols_,
    )

## 2. ArcSinh-transform the gated marker columns

Transform each nonmetadata marker in `df_gated` as `arcsinh(value / 5)`. `df_arcsinh` preserves the input columns and row order.

In [ ]:
df_arcsinh, arcsinh_columns = arcsinh_transform(
    df_batch_corrected.copy(), skip_columns=METADATA_COLUMNS
)
if set(df_arcsinh.columns) != set(arcsinh_columns + METADATA_COLUMNS):
    print("ERROR: ArcSinh changed the dataframe columns.")
print("Pooled ArcSinh table shape:", df_arcsinh.shape)

## 4. Normalize marker values

Normalize each input CSV independently using source paths keyed by `cell_uid`. Select every numeric marker except metadata and control markers; all rows are retained in their original order.

In [ ]:
df_normalized, normalized_columns = normalize_pooled_dataframe_new(
    df_arcsinh.copy(),
    skip_columns=METADATA_COLUMNS,
)
if not df_normalized.columns.equals(df_arcsinh.columns):
    print("ERROR: normalization changed the dataframe columns.")
print("Pooled normalized table shape:", df_normalized.shape)

## 5. Z-score marker columns from a balanced reference

- `df_normalized` contains GFP-high cells; no `GFP_high` column is required.
- By default, balance untreated and treated cells across `experiment`, then `sample_name`. Set `use_labels=True` to also balance basal and classical cells; rows missing `labels` are dropped in that mode.
- The function defaults to a maximum reference size of 10,000 cells per group and reduces it when availability requires. It reads the random seed from `RANDOM_STATE` in `.env`.
- Fit one `StandardScaler` on the combined reference and transform every non-metadata feature in the retained rows.
- Return the scaled dataframe, selected and dropped row indices, and a summary with `n` and `n_subsample`; aggregate rows have no `sample_name`.

In [ ]:
df_scaled, scaler, reference_indices, dropped_cell_indices, scaling_summary = hierarchical_zscore(
    df_normalized.copy(),
    use_labels=False,
)
# expected_index = (
#     df_normalized.index[df_normalized["labels"].notna()]
#     if use_labels
#     else df_normalized.index
# )
# if not df_scaled.columns.equals(df_normalized.columns) or not df_scaled.index.equals(expected_index):
#     raise ValueError("Reference scaling changed the expected input rows or columns.")


print(f"Reference cells selected: {len(reference_indices)}")
print(f"Cells dropped for missing labels: {len(dropped_cell_indices)}")
print("Pooled z-scored table shape:", df_scaled.shape)
display(scaling_summary)

## 6. Subsample and save data

- `df_scaled` already contains GFP-high cells.
- When `SUBSAMPLE` is true, select equal sample sizes within each experiment, balancing by condition and sample; `use_labels=True` also balances subtype labels.
- Build the summary with the same grouping mode, including per-sample counts and aggregate counts without `sample_name`.
- Save the selected data, summary, and complete z-scored input.

In [ ]:
import pandas as pd

df_subsampled, summary = subsample_gfp_high(
    df_scaled.copy(),
    subsample=SUBSAMPLE,
    use_labels=use_labels,
)


GFP_HIGH_OUT = OUTPUT_ROOT / "preprocess"
GFP_HIGH_OUT.mkdir(parents=True, exist_ok=True)
print(f"Subsampling enabled: {SUBSAMPLE}")
print(f"Label balancing enabled: {use_labels}")
print("Selected z-scored GFP_high table shape:", df_subsampled.shape)
with pd.option_context("display.max_rows", None):
    display(summary)


SUFFIX1 = "" if SKIP_BATCH_CORRECTION else "_BATCH_CORRECTED"
SUFFIX2 = SUFFIX1 + (
    "" if not SUBSAMPLE else ("_BALANCED_LABELS" if use_labels else "_BALANCED_TREATMENT")
)

df_subsampled.to_csv(GFP_HIGH_OUT / f"GFP_high_ZSCORED{SUFFIX2}.csv", index=False)  # subsampled
print(f"GFP_high_ZSCORED{SUFFIX2} shape:", df_subsampled.shape)

df_scaled.to_csv(GFP_HIGH_OUT / f"all_GFP_high_ZSCORED{SUFFIX1}.csv", index=False)  # all cells
print(f"all_GFP_high_ZSCORED{SUFFIX1} shape:", df_scaled.shape)

summary.to_csv(GFP_HIGH_OUT / f"GFP_high_selection_summary{SUFFIX2}.csv", index=False)